In [ ]:
import os

# Unzip the dataset using the shell command
zip_path = '/content/Data.zip'
extract_path = '/content/Data'

# Ensure the extraction directory exists
os.makedirs(extract_path, exist_ok=True)

#unzip
!unzip -o -q {zip_path} -d {extract_path}

print(f"Data extracted to {extract_path}")

# Verify content
print(f"Contents of {extract_path}:")
!ls -F {extract_path}

In [ ]:

import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tensorflow.keras import layers, models, Input
from tensorflow.keras.applications import (
    ResNet50, ResNet101V2,
    MobileNetV2, MobileNetV3Small,
    EfficientNetB0, EfficientNetB4,
    Xception, InceptionV3
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

# Setup directories in Colab
os.makedirs('training_plots', exist_ok=True)
os.makedirs('confusion_matrices', exist_ok=True)
os.makedirs('models', exist_ok=True)

# 1. Dataset preparation
def prepare_dataset(base_path, categories, test_size=0.2, val_size=0.1):
    image_paths = []
    labels = []

    # Often zip files create a nested folder
    actual_base = base_path
    for root, dirs, files in os.walk(base_path):
        if all(cat in dirs for cat in categories):
            actual_base = root
            break

    for category in categories:
        category_path = os.path.join(actual_base, category)
        if not os.path.exists(category_path):
            print(f"Warning: Folder {category} not found in {actual_base}")
            continue
        for img in os.listdir(category_path):
            if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(os.path.join(category_path, img))
                labels.append(category)

    le = LabelEncoder()
    y_encoded = le.fit_transform(labels)

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        image_paths, y_encoded, test_size=test_size, stratify=y_encoded, random_state=42)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=val_size/(1-test_size),
        stratify=y_train_val, random_state=42)

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), le.classes_

# Configuration
categories = ['Mild Dementia', 'Moderate Dementia', 'Non Demented', 'Very mild Dementia']
base_data_dir = '/content/Data'

(X_train, y_train), (X_val, y_val), (X_test, y_test), class_names = prepare_dataset(base_data_dir, categories)

#Graphs and figure generation
def plot_pr_curve(y_test, y_proba, class_names, model_name):
    plt.figure(figsize=(10, 8))
    y_test_oh = tf.one_hot(y_test, depth=len(class_names)).numpy()
    for i in range(len(class_names)):
        precision, recall, _ = precision_recall_curve(y_test_oh[:, i], y_proba[:, i])
        ap = average_precision_score(y_test_oh[:, i], y_proba[:, i])
        plt.plot(recall, precision, label=f'{class_names[i]} (AP={ap:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'PR Curve: {model_name}')
    plt.legend(loc='best')
    plt.grid(alpha=0.3)
    plt.savefig(f'training_plots/{model_name}_PR_Curve.png')
    plt.show()

def plot_roc_curve(y_test, y_proba, class_names, model_name):
    plt.figure(figsize=(10, 8))
    y_test_oh = tf.one_hot(y_test, depth=len(class_names)).numpy()
    for i in range(len(class_names)):
        fpr, tpr, _ = roc_curve(y_test_oh[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'ROC Curve: {model_name}')
    plt.legend(loc="lower right")
    plt.savefig(f'training_plots/{model_name}_ROC_Curve.png')
    plt.show()

# 2. Data pipeline
def create_dataset(X, y, batch_size=16):
    def parse_image(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, [224, 224])
        return image, label

    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = create_dataset(X_train, y_train)
val_ds = create_dataset(X_val, y_val)
test_ds = create_dataset(X_test, y_test)

# Define the 8 models to compare
model_factories = {
    ""
    "ResNet50": (ResNet50, tf.keras.applications.resnet50.preprocess_input),
    "ResNet101V2": (ResNet101V2, tf.keras.applications.resnet_v2.preprocess_input),
    "MobileNetV2": (MobileNetV2, tf.keras.applications.mobilenet_v2.preprocess_input),
    "MobileNetV3Small": (MobileNetV3Small, tf.keras.applications.mobilenet_v3.preprocess_input),
    "EfficientNetB0": (EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input),
    "EfficientNetB4": (EfficientNetB4, tf.keras.applications.efficientnet.preprocess_input),
    "Xception": (Xception, tf.keras.applications.xception.preprocess_input),
    "InceptionV3": (InceptionV3, tf.keras.applications.inception_v3.preprocess_input)
}

def build_dynamic_model(model_name, base_model_class, preprocess_fn, input_shape=(224, 224, 3), num_classes=4):
    inputs = Input(shape=input_shape)
    # Apply the specific preprocessing for this architecture
    x = layers.Lambda(preprocess_fn)(inputs)

    base_model = base_model_class(include_top=False, weights='imagenet', input_shape=input_shape)
    base_model.trainable = True # Start with frozen weights for stability

    x = base_model(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs)

# 4. Training
all_results = []

for name, (model_class, preprocess_fn) in model_factories.items():
    print(f"\n{'='*20}\nTraining Model: {name}\n{'='*20}")

    # 1. Build and Compile
    model = build_dynamic_model(name, model_class, preprocess_fn)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # 2. Train
    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(f'models/best_{name}.h5', monitor='val_accuracy', save_best_only=True)
    ]

    model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks, verbose=1)

    # 3. Predict & Evaluate
    y_proba = model.predict(test_ds)
    y_pred = y_proba.argmax(axis=1)

    # 4. Save Confusion Matrix
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix: {name}")
    plt.savefig(f'confusion_matrices/{name}_cm.png')
    plt.close() # Close to save memory

    # 5. Store Metrics
    res = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted'),
        'F1': f1_score(y_test, y_pred, average='weighted')
    }
    all_results.append(res)

    # Print metrics for the current model
    print(f"\n--- Metrics for {name} ---")
    print(f"Accuracy: {res['Accuracy']:.4f}")
    print(f"Precision: {res['Precision']:.4f}")
    print(f"Recall: {res['Recall']:.4f}")
    print(f"F1-Score: {res['F1']:.4f}")
    print("--------------------------")

    # Run your custom PR/ROC functions
    plot_pr_curve(y_test, y_proba, class_names, name)
    plot_roc_curve(y_test, y_proba, class_names, name)

# Final Comparison Table
df_comparison = pd.DataFrame(all_results)
print("\n--- Final Model Comparison ---")
print(df_comparison.sort_values(by='F1', ascending=False))

# Save the comparison table to an Excel file
excel_output_path = 'model_comparison_metrics.xlsx'
df_comparison.to_excel(excel_output_path, index=False)
print(f"\nModel comparison metrics saved to {excel_output_path}")